# Demo pipeline Legal Contract Analyzer

Notebook demonstrativ pentru partea de orchestrare, recomandari si raport final. Ruleaza end-to-end workflow-ul pe un contract PDF si afiseaza rezultatele generate in `logs/` si `data/`.


In [ ]:
import os
from pathlib import Path
from IPython.display import Markdown, Image, display

os.chdir(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd())
print(Path.cwd())


## Contract ales

Pentru demo se poate folosi `data/contract_exemplu.pdf`. Daca fisierul nu exista sau parserul nu extrage clauze, workflow-ul foloseste un fallback local, astfel incat diagrama, raportul si interfata sa poata fi demonstrate fara configurari suplimentare.


In [ ]:
from src.graph.workflow import run_pipeline

pdf_path = 'data/contract_exemplu.pdf'
state = run_pipeline(pdf_path, retrieval_threshold=0.5, high_risk_threshold=2)
state.keys()


In [ ]:
parsed = state.get('parsed_doc')
print('Clauze:', len(parsed.clauses))
print('High risk alert:', state.get('high_risk_alert'))
print('Raport:', state.get('report_path'))
print('Iteratie:', state.get('iteration'))


## Timpi per nod


In [ ]:
import pandas as pd
pd.DataFrame(state.get('run_log', []))


## Vizualizari generate


In [ ]:
for image_path in ['logs/retrieval_heatmap.png', 'logs/risk_distribution.png', 'logs/workflow_graph.png']:
    if Path(image_path).exists():
        print(image_path)
        display(Image(filename=image_path))
    else:
        print(f'{image_path} nu exista in aceasta rulare.')


## Raport final


In [ ]:
report_path = state.get('report_path')
if report_path and Path(report_path).exists():
    display(Markdown(Path(report_path).read_text(encoding='utf-8')))
else:
    print('Raportul nu a fost generat.')


## Concluzie

Workflow-ul demonstreaza lantul parse_document -> retrieve_context -> assess_risk -> quality_check -> flag_high_risk -> generate_recommendations -> compile_report. Costul estimat depinde de numarul de clauze si de disponibilitatea cheii OpenAI; in modul fallback costul este 0 USD. O limitare reala este dependenta calitatii raspunsului de calitatea corpusului si de corectitudinea chunk-urilor recuperate. O imbunatatire posibila este adaugarea unui reranker si a unui set etichetat manual pentru calculul preciziei, recall-ului si F1.
